In [ ]:
# Luong (Multiplicative) Attention
This notebook implements the "General" variant of Luong Attention.
It uses matrix multiplication (dot products) which makes it computationally faster than Bahdanau.

In [ ]:
# %%
import tensorflow as tf

class LuongAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(LuongAttention, self).__init__()
        # W layer aligns the Encoder states to the Decoder state dimensions
        self.W = tf.keras.layers.Dense(units)

    def call(self, query, values):
        # Luong expects the query to already have a time dimension. If not, expand it.
        if len(query.shape) == 2:
            query = tf.expand_dims(query, 1)

        # 1. Calculate the Score (Dot Product)
        # Pass values through the Dense layer to align dimensions
        transformed_values = self.W(values)
        
        # Matrix multiplication: query * W(values)
        score = tf.matmul(query, transformed_values, transpose_b=True)

        # 2. Calculate Attention Weights (Softmax)
        attention_weights = tf.nn.softmax(score, axis=-1)

        # 3. Create Context Vector
        context_vector = tf.matmul(attention_weights, values)
        
        # Remove the extra time dimension to match standard output style
        context_vector = tf.squeeze(context_vector, axis=1)
        attention_weights = tf.squeeze(attention_weights, axis=1)

        return context_vector, attention_weights

In [ ]:
# %%
# --- TEST THE LUONG LAYER ---
print("🧪 Testing Luong Attention")

# Dummy data: Batch=1, Time Steps=5, Hidden Size=10
sample_query = tf.random.normal((1, 10))
sample_values = tf.random.normal((1, 5, 10))

attention_layer = LuongAttention(units=10)
context, weights = attention_layer(sample_query, sample_values)

print(f"Query Shape: {sample_query.shape}")
print(f"Values (Encoder States) Shape: {sample_values.shape}")
print(f"Context Vector Shape: {context.shape}")
print(f"Attention Weights Shape: {weights.shape}")